# Phase 12 — Estimation against matched controls

Dynamic event studies with never-retracted authors as the omitted reference
group, so every coefficient is an absolute difference from matched controls.

**Inputs:** `data/interim/phase11_panel_controls.csv`,
`data/interim/phase11_panel_citations.csv`

**Outputs**

| File | Contents |
|---|---|
| `data/results/phase12_results.csv` | coefficients, all outcomes |
| `data/results/phase12_tost.csv` | equivalence tests on the arm contrasts |

Year fixed effects are on here (control pseudo-retraction years break the
calendar-year/event-time collinearity and let control rows identify the
counterfactual). Two event times, -2 and -1, are omitted rather than one; the
condition number is reported with each specification.

In [1]:
import os
import math
import itertools
import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

PANEL = "data/interim/phase11_panel_controls.csv"
PANEL_CIT = "data/interim/phase11_panel_citations.csv"

OUT_RESULTS = "data/results/phase12_results.csv"
OUT_TOST = "data/results/phase12_tost.csv"

ARMS = ["AUTHOR_MISCONDUCT", "HONEST_ERROR", "EDITORIAL_COMPROMISE"]
REF_ARM = "CONTROL"

REF_EVENTS = (-2, -1)
ANALYSIS_PRE, ANALYSIS_POST = 6, 6
CIT_ANALYSIS_PRE, CIT_ANALYSIS_POST = 3, 3
HORIZONS = (1, 3, 6)

YEAR_FE = True
CLUSTER_VAR = "cluster_id"

# Declared before estimation.
MIN_MEANINGFUL_EXIT = 0.02
MIN_MEANINGFUL_PUBS = 0.05

# Monte Carlo variation across draws in the estimator validation, worst in the
# smallest arm.
MC_VARIATION = 0.15

pd.set_option("display.width", 220)
os.makedirs("data/results", exist_ok=True)

print(f"reference group    {REF_ARM}")
print(f"year fixed effects {YEAR_FE}")
print(f"clustering on      {CLUSTER_VAR}")
print(f"thresholds         exit {MIN_MEANINGFUL_EXIT}, counts {MIN_MEANINGFUL_PUBS}")

reference group    CONTROL
year fixed effects True
clustering on      cluster_id
thresholds         exit 0.02, counts 0.05


## Estimation engine

In [2]:
def _norm_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def absorb_fe(df, cols, groups, tol=1e-10, max_iter=100):
    """Alternating projections run to convergence."""
    out = df[cols].astype(float).copy()
    for _ in range(max_iter):
        prev = out.values.copy()
        for g in groups:
            gg = df[g]
            for c in cols:
                out[c] = out[c] - out[c].groupby(gg).transform("mean")
        if np.abs(out.values - prev).max() < tol:
            break
    return out


def ols_cluster(y, X, cluster, names):
    y = np.asarray(y, float); X = np.asarray(X, float)
    n, k = X.shape
    XtX_inv = np.linalg.pinv(X.T @ X)
    beta = XtX_inv @ (X.T @ y)
    resid = y - X @ beta

    cl = pd.Series(cluster).astype(str).values
    order = np.argsort(cl)
    cl_s, X_s, r_s = cl[order], X[order], resid[order]
    bounds = np.flatnonzero(np.r_[True, cl_s[1:] != cl_s[:-1], True])
    meat = np.zeros((k, k))
    for a, b in zip(bounds[:-1], bounds[1:]):
        s = X_s[a:b].T @ r_s[a:b]
        meat += np.outer(s, s)

    G = len(bounds) - 1
    dof = (G / max(G - 1, 1)) * ((n - 1) / max(n - k, 1))
    V = dof * (XtX_inv @ meat @ XtX_inv)
    se = np.sqrt(np.clip(np.diag(V), 0, None))
    with np.errstate(divide="ignore", invalid="ignore"):
        t = np.where(se > 0, beta / se, np.nan)
    p = np.array([2 * (1 - _norm_cdf(abs(ti))) if np.isfinite(ti) else np.nan
                  for ti in t])

    res = pd.DataFrame({"term": names, "coef": beta, "se": se, "t": t, "p": p,
                        "ci_lo": beta - 1.96 * se, "ci_hi": beta + 1.96 * se})
    res.attrs["n_clusters"] = G
    res.attrs["vcov"] = V
    res.attrs["names"] = names
    return res


def check_design(X):
    """Flag a rank-deficient or ill-conditioned design; pinv would otherwise
    return a minimum-norm solution for a singular system."""
    if np.linalg.matrix_rank(X) < X.shape[1]:
        return False, "rank-deficient", np.inf
    c = float(np.linalg.cond(X.T @ X))
    return (c <= 1e10), ("" if c <= 1e10 else "ill-conditioned"), c


def estimate(panel, outcome, label, pre=None, post=None,
             active_only=False, year_fe=YEAR_FE):
    """Event study with REF_ARM omitted. Returns coefficients and the vcov."""
    pre = ANALYSIS_PRE if pre is None else pre
    post = ANALYSIS_POST if post is None else post

    d = panel.dropna(subset=[outcome, CLUSTER_VAR, "arm"]).copy()
    d = d[(d.event_time >= -pre) & (d.event_time <= post)]

    if active_only:
        n0 = len(d)
        d = d[d.active == 1]
        print(f"  active author-years: {n0:,} -> {len(d):,} ({len(d)/n0:.1%})")

    groups = [g for g in ARMS if g in set(d.arm)]
    if not groups:
        return None

    ks = [k for k in sorted(d.event_time.unique()) if k not in REF_EVENTS]
    ev = pd.DataFrame({f"k{k:+d}": (d.event_time == k).astype(float)
                       for k in ks}, index=d.index)

    parts, names = [], []
    for g in groups:
        m = (d.arm == g).astype(float).values[:, None]
        parts.append(ev.values * m)
        names += [f"{c}:{g}" for c in ev.columns]
    X = np.hstack(parts)
    y = d[outcome].values.astype(float)

    fe = ["author_id"] + (["year"] if year_fe else [])
    tmp = pd.DataFrame(X, columns=[f"c{i}" for i in range(X.shape[1])],
                       index=d.index)
    tmp[outcome] = y
    for g in fe:
        tmp[g] = d[g].values
    ab = absorb_fe(tmp, [f"c{i}" for i in range(X.shape[1])] + [outcome], fe)
    X, y = ab[[f"c{i}" for i in range(X.shape[1])]].values, ab[outcome].values

    ok, why, cond = check_design(X)
    print(f"  {label}: {len(d):,} rows | {d.author_id.nunique():,} authors "
          f"| cond(X'X) {cond:.2e}")
    if not ok:
        print(f"    [!] design {why}; not estimated")
        return None

    res = ols_cluster(y, X, d[CLUSTER_VAR].values, names)
    parsed = res.term.str.extract(r"^k([+-]?\d+):(.+)$")
    res["event_time"] = pd.to_numeric(parsed[0], errors="coerce")
    res["group"] = parsed[1]
    res["outcome"] = label
    res["n_obs"] = len(d)
    res["n_authors"] = d.author_id.nunique()
    res["n_clusters"] = res.attrs["n_clusters"]
    res["cond"] = cond
    print(f"    {res.attrs['n_clusters']:,} clusters")

    out = res.dropna(subset=["event_time"]).sort_values(["group", "event_time"])
    out.attrs["vcov"] = res.attrs["vcov"]
    out.attrs["names"] = res.attrs["names"]
    return out

## Reporting

In [3]:
def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""


def parallel_trends(res, label):
    """Pre-period slopes per arm. The slope is the quantity the arm contrast
    rests on; the significance count moves with the standard errors."""
    rows = []
    for g, sub in res[res.event_time < 0].groupby("group"):
        sub = sub.sort_values("event_time")
        slope = (float(np.polyfit(sub.event_time, sub.coef, 1)[0])
                 if len(sub) >= 2 else np.nan)
        rows.append({"group": g, "n_coef": len(sub),
                     "n_sig": int((sub.p < 0.05).sum()),
                     "mean_abs": round(float(sub.coef.abs().mean()), 4),
                     "slope_per_year": round(slope, 4)})
    t = pd.DataFrame(rows)
    if t.empty:
        return
    print(f"\npre-period slopes — {label}")
    print(t.to_string(index=False))
    if t.slope_per_year.notna().sum() > 1:
        spread = float(t.slope_per_year.max() - t.slope_per_year.min())
        print(f"  spread of slopes across arms: {spread:.4f}")


def report(res, label, threshold=None, horizons=HORIZONS):
    d = res.copy()
    d["sig"] = d.p.apply(stars)
    sel = d[d.event_time.isin(horizons)]
    print(f"\n{label} — absolute effects at {horizons}")
    print(sel[["group", "event_time", "coef", "se", "p", "sig",
               "ci_lo", "ci_hi"]].round(4).to_string(index=False))

    last = max(horizons)
    at = d[d.event_time == last]
    if len(at) > 1:
        spread = float(at.coef.max() - at.coef.min())
        print(f"  spread across arms at +{last}: {spread:.4f}")

## Load

In [4]:
panel = pd.read_csv(PANEL, low_memory=False)
print(f"panel  {len(panel):,} rows, {panel.author_id.nunique():,} authors")
print(panel.drop_duplicates("author_id").arm.value_counts().to_string())

if CLUSTER_VAR not in panel.columns:
    raise SystemExit(f"{CLUSTER_VAR} not in the panel; rerun Phase 11")

t_cl = set(panel.loc[panel.arm != REF_ARM, CLUSTER_VAR])
c_cl = set(panel.loc[panel.arm == REF_ARM, CLUSTER_VAR])
print(f"\nclusters: {len(t_cl):,} treated, {len(c_cl):,} control, "
      f"{len(t_cl & c_cl):,} shared")

panel_cit = None
if os.path.isfile(PANEL_CIT):
    panel_cit = pd.read_csv(PANEL_CIT, low_memory=False)
    print(f"citation panel  {len(panel_cit):,} rows, "
          f"{panel_cit.author_id.nunique():,} authors")

results = []
vcovs = {}

panel  475,189 rows, 36,553 authors
arm
CONTROL                 19259
AUTHOR_MISCONDUCT        9617
HONEST_ERROR             4894
EDITORIAL_COMPROMISE     2783

clusters: 2,423 treated, 2,029 control, 1,245 shared
citation panel  336,784 rows, 48,112 authors


## Exit from publishing

In [5]:
r = estimate(panel, "active", "active")
if r is not None:
    results.append(r)
    vcovs["active"] = (r.attrs["vcov"], r.attrs["names"])
    parallel_trends(r, "active")
    report(r, "EXIT", MIN_MEANINGFUL_EXIT)

  active: 475,189 rows | 36,553 authors | cond(X'X) 2.29e+01
    3,207 clusters

pre-period slopes — active
               group  n_coef  n_sig  mean_abs  slope_per_year
   AUTHOR_MISCONDUCT       4      3    0.0373          0.0173
EDITORIAL_COMPROMISE       4      4    0.0856          0.0236
        HONEST_ERROR       4      4    0.0612          0.0214
  spread of slopes across arms: 0.0063

EXIT — absolute effects at (1, 3, 6)
               group  event_time    coef     se      p sig   ci_lo   ci_hi
   AUTHOR_MISCONDUCT           1 -0.0475 0.0061 0.0000 *** -0.0595 -0.0354
   AUTHOR_MISCONDUCT           3 -0.0446 0.0079 0.0000 *** -0.0601 -0.0290
   AUTHOR_MISCONDUCT           6 -0.0286 0.0087 0.0010  ** -0.0458 -0.0115
EDITORIAL_COMPROMISE           1 -0.0329 0.0130 0.0114   * -0.0584 -0.0074
EDITORIAL_COMPROMISE           3 -0.0264 0.0121 0.0293   * -0.0502 -0.0027
EDITORIAL_COMPROMISE           6 -0.0173 0.0128 0.1768     -0.0423  0.0078
        HONEST_ERROR           1 -0.0306 0

## Publication output, both margins

The intensive margin conditions on an outcome the treatment affects, so the two
margins are not additive and are not subtracted from one another.

In [6]:
r = estimate(panel, "publications", "publications")
if r is not None:
    results.append(r)
    vcovs["publications"] = (r.attrs["vcov"], r.attrs["names"])
    parallel_trends(r, "publications")
    report(r, "PUBLICATIONS, combined margin")

r = estimate(panel, "publications", "publications_active_only",
             active_only=True)
if r is not None:
    results.append(r)
    vcovs["publications_active_only"] = (r.attrs["vcov"], r.attrs["names"])
    parallel_trends(r, "publications_active_only")
    report(r, "PUBLICATIONS, intensive margin")

  publications: 475,189 rows | 36,553 authors | cond(X'X) 2.29e+01
    3,207 clusters

pre-period slopes — publications
               group  n_coef  n_sig  mean_abs  slope_per_year
   AUTHOR_MISCONDUCT       4      4    0.5948          0.1685
EDITORIAL_COMPROMISE       4      4    1.3819          0.3523
        HONEST_ERROR       4      3    0.8015          0.1561
  spread of slopes across arms: 0.1962

PUBLICATIONS, combined margin — absolute effects at (1, 3, 6)
               group  event_time    coef     se      p sig   ci_lo   ci_hi
   AUTHOR_MISCONDUCT           1 -0.2056 0.1029 0.0457   * -0.4073 -0.0039
   AUTHOR_MISCONDUCT           3  0.0188 0.1610 0.9068     -0.2968  0.3344
   AUTHOR_MISCONDUCT           6  0.7871 0.2912 0.0069  **  0.2163  1.3578
EDITORIAL_COMPROMISE           1  0.0634 0.2597 0.8070     -0.4455  0.5724
EDITORIAL_COMPROMISE           3  0.4417 0.2799 0.1145     -0.1069  0.9903
EDITORIAL_COMPROMISE           6  0.6497 0.5808 0.2633     -0.4886  1.7881
     

## Margin decomposition

In [7]:
def coef_at(label, arm, k):
    for res in results:
        if res.outcome.iloc[0] != label:
            continue
        m = res[(res.group == arm) & (res.event_time == k)]
        if len(m):
            return float(m.coef.iloc[0]), float(m.p.iloc[0])
    return np.nan, np.nan


rows = []
for arm in ARMS:
    for k in HORIZONS:
        exit_c, _ = coef_at("active", arm, k)
        comb, p_c = coef_at("publications", arm, k)
        act, p_a = coef_at("publications_active_only", arm, k)
        if np.isnan(exit_c):
            continue
        rows.append({"arm": arm, "k": k, "exit": round(exit_c, 4),
                     "combined": round(comb, 4), "p_comb": round(p_c, 4),
                     "if_active": round(act, 4), "p_act": round(p_a, 4)})

margin_decomp = pd.DataFrame(rows)
if not margin_decomp.empty:
    print(margin_decomp.to_string(index=False))

                 arm  k    exit  combined  p_comb  if_active  p_act
   AUTHOR_MISCONDUCT  1 -0.0475   -0.2056  0.0457    -0.1152 0.3570
   AUTHOR_MISCONDUCT  3 -0.0446    0.0188  0.9068     0.1747 0.3667
   AUTHOR_MISCONDUCT  6 -0.0286    0.7871  0.0069     1.2623 0.0009
        HONEST_ERROR  1 -0.0306    0.0389  0.8024     0.0731 0.6897
        HONEST_ERROR  3 -0.0327    0.3785  0.0422     0.5139 0.0229
        HONEST_ERROR  6 -0.0251    0.8760  0.0074     1.1687 0.0056
EDITORIAL_COMPROMISE  1 -0.0329    0.0634  0.8070     0.0981 0.7350
EDITORIAL_COMPROMISE  3 -0.0264    0.4417  0.1145     0.5846 0.0610
EDITORIAL_COMPROMISE  6 -0.0173    0.6497  0.2633     0.8951 0.1749


## Equivalence tests

Whether the arms differ by more than the pre-declared bound, by two one-sided
tests. A contrast between two arms comes from one regression, so its standard
error uses the covariance between them.

In [8]:
def contrast(vcov, names, a, b, k):
    """coef(a) - coef(b) at event time k, with the covariance-correct SE."""
    ia = names.index(f"k{k:+d}:{a}") if f"k{k:+d}:{a}" in names else None
    ib = names.index(f"k{k:+d}:{b}") if f"k{k:+d}:{b}" in names else None
    if ia is None or ib is None:
        return None
    var = vcov[ia, ia] + vcov[ib, ib] - 2 * vcov[ia, ib]
    return ia, ib, float(np.sqrt(max(var, 0)))


def tost(diff, se, bound):
    """Two one-sided tests; p is the larger of the two."""
    if se <= 0:
        return np.nan
    p_lo = 1 - _norm_cdf((diff + bound) / se)
    p_hi = _norm_cdf((diff - bound) / se)
    return float(max(p_lo, p_hi))


tost_rows = []
for label, (V, names) in vcovs.items():
    bound = (MIN_MEANINGFUL_EXIT if label == "active"
             else MIN_MEANINGFUL_PUBS)
    res = next(r for r in results if r.outcome.iloc[0] == label)
    coefs = {(row.group, int(row.event_time)): float(row.coef)
             for row in res.itertuples()}

    for a, b in itertools.combinations(ARMS, 2):
        for k in HORIZONS:
            c = contrast(V, names, a, b, k)
            if c is None or (a, k) not in coefs or (b, k) not in coefs:
                continue
            _, _, se = c
            diff = coefs[(a, k)] - coefs[(b, k)]
            lo, hi = diff - 1.645 * se, diff + 1.645 * se
            p = tost(diff, se, bound)
            tost_rows.append({
                "outcome": label,
                "contrast": f"{a.split('_')[0][:4]} - {b.split('_')[0][:4]}",
                "k": k, "diff": round(diff, 4), "se": round(se, 4),
                "ci90_lo": round(lo, 4), "ci90_hi": round(hi, 4),
                "bound": bound, "p_tost": round(p, 4) if np.isfinite(p) else np.nan,
                "equivalent": bool(abs(lo) < bound and abs(hi) < bound),
                # tightest bound this contrast would clear; reported so the
                # declared bound is not moved after seeing the result.
                "min_bound": round(max(abs(lo), abs(hi)), 4),
            })

tost_df = pd.DataFrame(tost_rows)

for label in tost_df.outcome.unique() if not tost_df.empty else []:
    sel = tost_df[tost_df.outcome == label]
    print(f"\n{label}   bound +/-{sel.bound.iloc[0]}")
    print(sel.drop(columns=["outcome"]).to_string(index=False))
    n_eq = int(sel.equivalent.sum())
    print(f"  {n_eq} of {len(sel)} contrasts within the bound; "
          f"tightest bound clearing all: {sel.min_bound.max():.4f}")


active   bound +/-0.02
   contrast  k    diff     se  ci90_lo  ci90_hi  bound  p_tost  equivalent  min_bound
AUTH - HONE  1 -0.0169 0.0087  -0.0311  -0.0027   0.02  0.3599       False     0.0311
AUTH - HONE  3 -0.0119 0.0084  -0.0256   0.0019   0.02  0.1649       False     0.0256
AUTH - HONE  6 -0.0035 0.0088  -0.0180   0.0109   0.02  0.0304        True     0.0180
AUTH - EDIT  1 -0.0145 0.0139  -0.0373   0.0083   0.02  0.3469       False     0.0373
AUTH - EDIT  3 -0.0181 0.0128  -0.0391   0.0028   0.02  0.4418       False     0.0391
AUTH - EDIT  6 -0.0114 0.0130  -0.0328   0.0101   0.02  0.2540       False     0.0328
HONE - EDIT  1  0.0024 0.0140  -0.0206   0.0253   0.02  0.1033       False     0.0253
HONE - EDIT  3 -0.0063 0.0131  -0.0278   0.0152   0.02  0.1469       False     0.0278
HONE - EDIT  6 -0.0078 0.0136  -0.0302   0.0145   0.02  0.1851       False     0.0302
  1 of 9 contrasts within the bound; tightest bound clearing all: 0.0391

publications   bound +/-0.05
   contrast  

## Citations by arm

Controls were not matched on citation volume, and author fixed effects absorb a
level difference but not a difference in growth rate. The pre-period is reported
so a divergence can be inspected.

In [9]:
if panel_cit is not None and "citations" in panel_cit.columns:
    r = estimate(panel_cit, "citations", "citations",
                 pre=CIT_ANALYSIS_PRE, post=CIT_ANALYSIS_POST)
    if r is not None:
        results.append(r)
        parallel_trends(r, "citations")
        report(r, "CITATIONS", horizons=(1, 2, 3))
else:
    print("citation panel unavailable; skipped")

  citations: 336,784 rows | 48,112 authors | cond(X'X) 1.42e+01
    3,612 clusters

pre-period slopes — citations
               group  n_coef  n_sig  mean_abs  slope_per_year
   AUTHOR_MISCONDUCT       1      0    4.0810             NaN
EDITORIAL_COMPROMISE       1      0   19.6175             NaN
        HONEST_ERROR       1      1   25.2302             NaN

CITATIONS — absolute effects at (1, 2, 3)
               group  event_time    coef      se      p sig    ci_lo    ci_hi
   AUTHOR_MISCONDUCT           1  4.4125  5.6686 0.4363      -6.6981  15.5230
   AUTHOR_MISCONDUCT           2  4.4782  7.1238 0.5296      -9.4845  18.4409
   AUTHOR_MISCONDUCT           3  5.0328  8.3318 0.5458     -11.2975  21.3632
EDITORIAL_COMPROMISE           1 50.7671 26.9896 0.0600      -2.1326 103.6668
EDITORIAL_COMPROMISE           2 63.3241 36.1484 0.0798      -7.5267 134.1750
EDITORIAL_COMPROMISE           3 62.1024 31.5115 0.0487   *   0.3399 123.8649
        HONEST_ERROR           1 62.2963 11.7932 

## Write

In [10]:
if results:
    for r in results:
        r.attrs.clear()
    out = pd.concat(results, ignore_index=True)
    out["meaningful"] = np.where(
        out.outcome == "active",
        out.coef.abs() >= MIN_MEANINGFUL_EXIT, np.nan)
    out.to_csv(OUT_RESULTS, index=False)
    print(f"{OUT_RESULTS}  {len(out):,} coefficient rows")
    print(out.outcome.value_counts().to_string())

if not tost_df.empty:
    tost_df.to_csv(OUT_TOST, index=False)
    print(f"\n{OUT_TOST}  {len(tost_df):,} contrasts")

data/results/phase12_results.csv  114 coefficient rows
outcome
active                      33
publications                33
publications_active_only    33
citations                   15

data/results/phase12_tost.csv  27 contrasts


---

Absolute differences from matched controls; reported alongside the Phase 9 relative estimates. Phase 13 re-estimates with a staggered-adoption estimator.